##In this notebook, we simulate rainfall-runoff accross 222 CAMELS-AUS catchments using the HyMoLAP model.

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize # USE IN THE MODEL CALIBRATION

from google.colab import files
import zipfile
import os

In [ ]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##CAMELS-DATA from my Google Drive.

In [ ]:
# ==============================
# Paths to ZIP files
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Data extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted in {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extraction
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 basins data
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (ET SILO)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Identify common stations
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip and s in stations_et and s in stations_Q
]

print(f"✅ Official common stations: {len(common_stations)}")

# ==============================
# 5️⃣ Subset common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]

# ==============================
# 6️⃣ Final verification
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("Stations (first 10):", common_stations[:10], "...")


✅ Directory /content/05_hydro already exists
✅ Directory /content/03_streamflow already exists
✅ 222 official basins loaded
✅ SILO precipitation: (43464, 224)
✅ SILO ET: (43464, 224)
✅ Streamflow: (23376, 224)
✅ Official common stations: 222
Precipitation: (43464, 222)
ET: (43464, 222)
Streamflow: (23376, 222)
Stations (first 10): ['912101A', '912105A', '915011A', '917107A', '919003A', '919201A', '919309A', '922101B', '925001A', '926002A'] ...


In [ ]:
# Verification of the periods
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())


Precipitation : 1900-01-01 00:00:00 → 2018-01-01 00:00:00
ET            : 1900-01-01 00:00:00 → 2018-01-01 00:00:00
Streamflow    : 1951-01-01 00:00:00 → 2014-01-01 00:00:00


In [ ]:
# ==============================
# 0️⃣ Reduce all series to the period 1 January 1980 → 31 December 2014
# ==============================
start_date = "1980-01-01"
end_date   = "2014-12-31"

precip = precip.loc[start_date:end_date]
et     = et.loc[start_date:end_date]
Q      = Q.loc[start_date:end_date]

# Verification
print("Common period verification:")
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())

Common period verification:
Precipitation : 1980-01-01 00:00:00 → 2014-01-01 00:00:00
ET            : 1980-01-01 00:00:00 → 2014-01-01 00:00:00
Streamflow    : 1980-01-01 00:00:00 → 2014-01-01 00:00:00


In [ ]:
# ==============================
# Diviser les séries en 70% calibration / 30% validation
# ==============================
# Vérifier l'index commun
common_index = precip.index.intersection(et.index).intersection(Q.index)

# Nombre total de jours
n_total = len(common_index)
n_cal   = int(n_total * 0.7)

# Dates de calibration et validation
cal_dates = common_index[:n_cal]
val_dates = common_index[n_cal:]

# Extraire les séries
precip_cal = precip.loc[cal_dates]
et_cal     = et.loc[cal_dates]
Q_cal      = Q.loc[cal_dates]

precip_val = precip.loc[val_dates]
et_val     = et.loc[val_dates]
Q_val      = Q.loc[val_dates]

# Affichage des périodes
print("Calibration period:", cal_dates.min().strftime('%d/%m/%Y'), "→", cal_dates.max().strftime('%d/%m/%Y'))
print("Validation period :", val_dates.min().strftime('%d/%m/%Y'), "→", val_dates.max().strftime('%d/%m/%Y'))

Calibration period: 01/01/1980 → 01/01/2003
Validation period : 01/01/2004 → 01/01/2014


GRHyMoLAP

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit

# ============================================
# 1️⃣ General parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1

stations = common_stations  # basins à traiter
results_HyMoLAP = {}

# ============================================
# 2️⃣ Metrics (Numba accelerated)
# ============================================

def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)


def NNSE(obs, sim):
    nse = NSE(obs, sim)
    if np.isnan(nse):
        return np.nan
    return 1 / (2 - nse)


def RMSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    return np.sqrt(np.mean((sim - obs)**2))


def PBIAS(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.sum(obs) == 0:
        return np.nan
    return  np.sum(sim - obs) / np.sum(obs)


def FHV(obs, sim, top_fraction=0.02):
    epsilon = 0
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_top = int(len(obs) * top_fraction)
    if n_top == 0:
        return np.nan
    idx = np.argsort(obs)[-n_top:]
    obs_top = obs[idx]
    sim_top = sim[idx]
    return  np.sum(sim_top - obs_top) / (np.sum(obs_top) + epsilon)


def FLV(obs, sim, bottom_fraction=0.3):
    epsilon = 1e-6
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_bot = int(len(obs) * bottom_fraction)
    if n_bot == 0:
        return np.nan
    idx = np.argsort(obs)[:n_bot]
    obs_bot = obs[idx]
    sim_bot = sim[idx]
    return  np.sum(sim_bot - obs_bot) / (np.sum(obs_bot) + epsilon)

# ============================================
# 3️⃣ Soil state X (Numba)
# ============================================

def state_soil(MU, LAMBDA, Pn):
    X = np.zeros(len(Pn))
    for t in range(1, len(Pn)):
        if Pn[t] == 0:
            X[t] = X[t-1] - (MU / LAMBDA) * X[t-1]
        else:
            X[t] = X[t-1] + (MU / LAMBDA) * Pn[t]
    return X

# ============================================
# 4️⃣ HyMoLAP Model (Numba)
# ============================================

def HyMoLAP_Model(params, Q0, Pn):
    MU, LAMBDA = params
    Q = np.zeros(len(Pn))
    Q[0] = Q0
    X = state_soil(MU, LAMBDA, Pn)
    for t in range(len(Pn)-1):
        Q[t+1] = max(
            0,
            Q[t] - (MU / LAMBDA) * Q[t]**(2*MU - 1) + (1 / LAMBDA) * X[t+1] * Pn[t+1]
        )
    return Q

# ============================================
# 5️⃣ Main loop over basins
# ============================================
for i, station_id in enumerate(stations, start=1):
    print(f"\n=== Station {station_id} ({i}/{len(stations)}) ===")

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    PET = et[station_id].loc[start_date:end_date].to_numpy(float)

    Pn = np.maximum(0, P - PET)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    missing_ratio = np.sum(np.isnan(Q_obs)) / N
    if missing_ratio > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    # ============================================
    # 6️⃣ Objective function for calibration
    # ============================================
    def objective(params, Q0, Pn_train, Q_obs_train):
        Q_sim = HyMoLAP_Model(params, Q0, Pn_train)
        nse = NSE(Q_obs_train, Q_sim)
        return 1 - nse if np.isfinite(nse) else 1e9

    # Multi-start optimization
    initial_guesses = [
        [1.0, 2], [0.6, 10], [0.7, 20], [0.7, 40],
        [1.4, 80], [1.8, 40], [1.8, 80], [1.8, 120],
        [1.0, 70], [1.0, 150]
    ]
    best_res = None
    best_val = np.inf
    for guess in initial_guesses:
        res = minimize(
            objective,
            guess,
            args=(Q0, Pn[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            options={"maxiter": 2500, "disp": False}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    MU, LAMBDA = best_res.x
    Qsim = HyMoLAP_Model([MU, LAMBDA], Q0, Pn)

    # ============================================
    # 7️⃣ Metrics calculation
    # ============================================
    NSE_cal = NSE(Q_obs[:b1], Qsim[:b1])
    NNSE_cal = NNSE(Q_obs[:b1], Qsim[:b1])
    RMSE_cal = RMSE(Q_obs[:b1], Qsim[:b1])
    PBIAS_cal = PBIAS(Q_obs[:b1], Qsim[:b1])
    FHV_cal = FHV(Q_obs[:b1], Qsim[:b1])
    FLV_cal = FLV(Q_obs[:b1], Qsim[:b1])

    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])
    NNSE_val = NNSE(Q_obs[b1:], Qsim[b1:])
    RMSE_val = RMSE(Q_obs[b1:], Qsim[b1:])
    PBIAS_val = PBIAS(Q_obs[b1:], Qsim[b1:])
    FHV_val = FHV(Q_obs[b1:], Qsim[b1:])
    FLV_val = FLV(Q_obs[b1:], Qsim[b1:])

    # ============================================
    # Display metrics
    # ============================================
    print(f"✅ Calibration → NSE: {NSE_cal:.3f}, NNSE: {NNSE_cal:.3f}, RMSE: {RMSE_cal:.3f}, PBIAS: {PBIAS_cal:.3f}")
    print(f"✅ Validation  → NSE: {NSE_val:.3f}, NNSE: {NNSE_val:.3f}, RMSE: {RMSE_val:.3f}, PBIAS: {PBIAS_val:.3f}")
    print(f"   FHV/FLV Cal: {FHV_cal:.3f} / {FLV_cal:.3f}, Val: {FHV_val:.3f} / {FLV_val:.3f}")
    print(f"   Params: MU={MU:.3f}, LAMBDA={LAMBDA:.3f}")

    # ============================================
    # 8️⃣ Store results
    # ============================================
    results_HyMoLAP[station_id] = {
        "params": {"MU": MU, "LAMBDA": LAMBDA},
        "NSE_cal": NSE_cal,
        "NNSE_cal": NNSE_cal,
        "RMSE_cal": RMSE_cal,
        "PBIAS_cal": PBIAS_cal,
        "FHV_cal": FHV_cal,
        "FLV_cal": FLV_cal,
        "NSE_val": NSE_val,
        "NNSE_val": NNSE_val,
        "RMSE_val": RMSE_val,
        "PBIAS_val": PBIAS_val,
        "FHV_val": FHV_val,
        "FLV_val": FLV_val,
        "Qsim": Qsim,
        "missing_ratio": missing_ratio
    }

print(f"\n✅ Finished: {len(results_HyMoLAP)} valid basins")


=== Station 912101A (1/222) ===
✅ Calibration → NSE: -0.305, NNSE: 0.434, RMSE: 0.850, PBIAS: 2.123
✅ Validation  → NSE: -0.023, NNSE: 0.494, RMSE: 1.246, PBIAS: 1.320
   FHV/FLV Cal: -0.821 / 17.355, Val: -0.832 / 19.821
   Params: MU=69.284, LAMBDA=283.732

=== Station 912105A (2/222) ===
✅ Calibration → NSE: -0.294, NNSE: 0.436, RMSE: 0.790, PBIAS: 1.878
✅ Validation  → NSE: -0.059, NNSE: 0.486, RMSE: 1.060, PBIAS: 1.098
   FHV/FLV Cal: -0.812 / 16.963, Val: -0.794 / 22.275
   Params: MU=12.960, LAMBDA=127.601

=== Station 915011A (3/222) ===
✅ Calibration → NSE: 0.140, NNSE: 0.538, RMSE: 1.185, PBIAS: 1.248
✅ Validation  → NSE: -0.084, NNSE: 0.480, RMSE: 1.051, PBIAS: 1.207
   FHV/FLV Cal: -0.763 / 698100165.102, Val: -0.802 / 406383820.300
   Params: MU=51.619, LAMBDA=290.260

=== Station 917107A (4/222) ===


/tmp/ipykernel_420/4258698500.py:111: RuntimeWarning: overflow encountered in scalar power
  Q[t] - (MU / LAMBDA) * Q[t]**(2*MU - 1) + (1 / LAMBDA) * X[t+1] * Pn[t+1]


✅ Calibration → NSE: 0.162, NNSE: 0.544, RMSE: 0.992, PBIAS: 1.274
✅ Validation  → NSE: -0.019, NNSE: 0.495, RMSE: 1.206, PBIAS: 0.525
   FHV/FLV Cal: -0.713 / 111.925, Val: -0.801 / 142.400
   Params: MU=5.415, LAMBDA=116.724

=== Station 919003A (5/222) ===


/tmp/ipykernel_420/4258698500.py:27: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)


✅ Calibration → NSE: 0.336, NNSE: 0.601, RMSE: 1.464, PBIAS: 1.444
✅ Validation  → NSE: 0.494, NNSE: 0.664, RMSE: 2.221, PBIAS: 0.547
   FHV/FLV Cal: -0.453 / 39.736, Val: -0.610 / 37.302
   Params: MU=1.557, LAMBDA=89.067

=== Station 919201A (6/222) ===
✅ Calibration → NSE: 0.266, NNSE: 0.577, RMSE: 2.691, PBIAS: 1.604
✅ Validation  → NSE: 0.310, NNSE: 0.592, RMSE: 3.425, PBIAS: 0.888
   FHV/FLV Cal: -0.546 / 1090.160, Val: -0.627 / 311.370
   Params: MU=1.548, LAMBDA=94.962

=== Station 919309A (7/222) ===
✅ Calibration → NSE: 0.328, NNSE: 0.598, RMSE: 1.142, PBIAS: 1.692
✅ Validation  → NSE: 0.518, NNSE: 0.675, RMSE: 1.741, PBIAS: 0.641
   FHV/FLV Cal: -0.428 / 3876.023, Val: -0.582 / 502.039
   Params: MU=1.680, LAMBDA=97.676

=== Station 922101B (8/222) ===
✅ Calibration → NSE: 0.665, NNSE: 0.749, RMSE: 3.530, PBIAS: 0.549
✅ Validation  → NSE: 0.640, NNSE: 0.735, RMSE: 3.395, PBIAS: 0.542
   FHV/FLV Cal: -0.305 / 53.882, Val: -0.257 / 11.439
   Params: MU=1.420, LAMBDA=54.364

==

...

✅ Calibration → NSE: -0.527, NNSE: 0.396, RMSE: 0.771, PBIAS: 4.402
✅ Validation  → NSE: -0.774, NNSE: 0.360, RMSE: 0.898, PBIAS: 11.323
   FHV/FLV Cal: -0.844 / 959664120.675, Val: -0.779 / 609516457.078
   Params: MU=34.542, LAMBDA=278.244

=== Station 407214 (60/222) ===
✅ Calibration → NSE: -0.279, NNSE: 0.439, RMSE: 0.741, PBIAS: 1.280
✅ Validation  → NSE: -0.117, NNSE: 0.472, RMSE: 0.963, PBIAS: 3.576
   FHV/FLV Cal: -0.852 / 195.121, Val: -0.780 / 590.544
   Params: MU=102.951, LAMBDA=416.378

=== Station 407215 (61/222) ===
✅ Calibration → NSE: -0.225, NNSE: 0.449, RMSE: 0.804, PBIAS: 0.964
✅ Validation  → NSE: -0.210, NNSE: 0.452, RMSE: 1.057, PBIAS: 2.845
   FHV/FLV Cal: -0.857 / 116.121, Val: -0.876 / 534365144.664
   Params: MU=65.201, LAMBDA=363.695

=== Station 407220 (62/222) ===
✅ Calibration → NSE: -0.260, NNSE: 0.442, RMSE: 0.812, PBIAS: 2.017
✅ Validation  → NSE: -0.403, NNSE: 0.416, RMSE: 0.856, PBIAS: 8.481
   FHV/FLV Cal: -0.860 / 285.979, Val: -0.778 / 341426062.

/tmp/ipykernel_420/4258698500.py:94: RuntimeWarning: invalid value encountered in scalar subtract
  X[t] = X[t-1] - (MU / LAMBDA) * X[t-1]


✅ Calibration → NSE: 0.346, NNSE: 0.605, RMSE: 1.424, PBIAS: 0.498
✅ Validation  → NSE: -0.228, NNSE: 0.449, RMSE: 1.258, PBIAS: 1.124
   FHV/FLV Cal: -0.654 / 19.921, Val: -0.668 / 47.360
   Params: MU=5.885, LAMBDA=46.101

=== Station 216002 (154/222) ===
✅ Calibration → NSE: 0.424, NNSE: 0.634, RMSE: 4.160, PBIAS: 2.029
✅ Validation  → NSE: -1.038, NNSE: 0.329, RMSE: 3.314, PBIAS: 3.514
   FHV/FLV Cal: -0.170 / 25.095, Val: 0.456 / 39.310
   Params: MU=1.307, LAMBDA=31.857

=== Station 216004 (155/222) ===
✅ Calibration → NSE: 0.403, NNSE: 0.626, RMSE: 2.835, PBIAS: 2.906
✅ Validation  → NSE: -1.778, NNSE: 0.265, RMSE: 2.167, PBIAS: 7.173
   FHV/FLV Cal: -0.147 / 238.465, Val: 0.441 / 151218.355
   Params: MU=1.422, LAMBDA=41.520

=== Station 218001 (156/222) ===
✅ Calibration → NSE: 0.503, NNSE: 0.668, RMSE: 2.007, PBIAS: 1.400
✅ Validation  → NSE: 0.403, NNSE: 0.626, RMSE: 1.714, PBIAS: 1.842
   FHV/FLV Cal: -0.291 / 17.067, Val: -0.074 / 30.416
   Params: MU=1.433, LAMBDA=43.970


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


✅ Calibration → NSE: -7.690, NNSE: 0.103, RMSE: 0.515, PBIAS: 8.802
✅ Validation  → NSE: -37.471, NNSE: 0.025, RMSE: 0.645, PBIAS: 32.248
   FHV/FLV Cal: -0.655 / 420.235, Val: -0.428 / 600066437.403
   Params: MU=33.419, LAMBDA=335.710

=== Station A2390523 (183/222) ===
✅ Calibration → NSE: -50.318, NNSE: 0.019, RMSE: 0.496, PBIAS: 12.612
✅ Validation  → NSE: -131.312, NNSE: 0.008, RMSE: 0.521, PBIAS: 23.329
   FHV/FLV Cal: 0.006 / 263.377, Val: 0.510 / 1152.558
   Params: MU=179.280, LAMBDA=508.844

=== Station A2390531 (184/222) ===
✅ Calibration → NSE: -0.037, NNSE: 0.491, RMSE: 0.139, PBIAS: -1.000
✅ Validation  → NSE: -0.019, NNSE: 0.495, RMSE: 0.087, PBIAS: -1.000
   FHV/FLV Cal: -1.000 / 0.000, Val: -1.000 / 0.000
   Params: MU=0.429, LAMBDA=2813.415

=== Station 604053 (185/222) ===
✅ Calibration → NSE: -2.676, NNSE: 0.214, RMSE: 0.507, PBIAS: 2.803
✅ Validation  → NSE: -11.853, NNSE: 0.072, RMSE: 0.650, PBIAS: 7.004
   FHV/FLV Cal: -0.715 / 231.253, Val: -0.582 / 766.393
   

NSE result summary

In [ ]:
# =============================================================
# 📌 FUNCTION TO PRINT METRIC SUMMARY
# =============================================================
def summarize_metric(metric_name, results_dict):

    values = [res.get(metric_name, np.nan) for res in results_dict.values()]
    values = np.array(values)
    values = values[~np.isnan(values)]  # remove NaN

    if len(values) == 0:
        print(f"{metric_name}: No valid values\n")
        return

    print(f"{metric_name}")
    print(f"Median : {np.percentile(values,50):.3f}")
    print(f"Mean   : {np.mean(values):.3f}")
    print(f"Min    : {np.min(values):.3f}")
    print(f"Max    : {np.max(values):.3f}")
    print(f"5th–95th percentile : {np.percentile(values,5):.3f} – {np.percentile(values,95):.3f}")
    print("-"*40)


# =============================================================
# 📌 CALIBRATION SUMMARY
# =============================================================
print("\n================= CALIBRATION SUMMARY =================\n")

metrics_cal = [
    "NSE_cal",
    "NNSE_cal",
    "RMSE_cal",
    "PBIAS_cal",
    "FHV_cal",
    "FLV_cal"
]

for metric in metrics_cal:
    summarize_metric(metric, results_HyMoLAP)


# =============================================================
# 📌 VALIDATION SUMMARY
# =============================================================
print("\n================= VALIDATION SUMMARY =================\n")

metrics_val = [
    "NSE_val",
    "NNSE_val",
    "RMSE_val",
    "PBIAS_val",
    "FHV_val",
    "FLV_val"
]

for metric in metrics_val:
    summarize_metric(metric, results_HyMoLAP)


================= CALIBRATION SUMMARY =================

NSE_cal
Median : 0.165
Mean   : -1.244
Min    : -121.883
Max    : 0.786
5th–95th percentile : -1.449 – 0.611
----------------------------------------
NNSE_cal
Median : 0.545
Mean   : 0.539
Min    : 0.008
Max    : 0.824
5th–95th percentile : 0.290 – 0.720
----------------------------------------
RMSE_cal
Median : 1.336
Mean   : 1.780
Min    : 0.073
Max    : 8.486
5th–95th percentile : 0.507 – 4.400
----------------------------------------
PBIAS_cal
Median : 0.943
Mean   : 1.574
Min    : -1.000
Max    : 29.183
5th–95th percentile : -0.113 – 3.814
----------------------------------------
FHV_cal
Median : -0.607
Mean   : -0.591
Min    : -1.000
Max    : 0.640
5th–95th percentile : -0.901 – -0.147
----------------------------------------
FLV_cal
Median : 27.438
Mean   : 148235698.034
Min    : 0.000
Max    : 2822875024.366
5th–95th percentile : 1.799 – 1050534537.504
----------------------------------------

================= VALIDATIO

In [ ]:
# ============================================
# Créer Excel avec validation metrics uniquement
# ============================================

records = []
for station_id, res in results_HyMoLAP.items():
    record = {
        "Station": station_id,
        "NSE_val": res["NSE_val"],
        "NNSE_val": res["NNSE_val"],
        "RMSE_val": res["RMSE_val"],
        "PBIAS_val": res["PBIAS_val"],
        "FHV_val": res["FHV_val"],
        "FLV_val": res["FLV_val"],
    }
    records.append(record)

df_val_metrics = pd.DataFrame(records)
excel_filename = "HyMoLAP_validation_metrics.xlsx"
df_val_metrics.to_excel(excel_filename, index=False)
print(f"✅ Excel saved: {excel_filename}")

# Télécharger si Colab
try:
    from google.colab import files
    files.download(excel_filename)
except:
    print("⚠️ Not running in Colab. File saved locally.")

✅ Excel saved: HyMoLAP_validation_metrics.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>